## 회원 CRUD - auth.users, user_details
- 삭제 : deleted_at에 날짜와 시간이 업데이트되면 소프트 삭제
- 조회시에는 deleted_at is not null로 조회될 수 있도록 한다.

In [ ]:
import os
import uuid
from datetime import datetime, timezone
from dotenv import load_dotenv
from supabase import create_client

# 환경변수 로드
load_dotenv()

url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_PUBLISHABLE_KEY")

# Supabase 클라이언트 생성
supabase = create_client(url, key)

In [ ]:
class UserService:

    def __init__(self):
        self.table = "user_details"

    # 회원 상세 정보 등록 (Create)
    def create_user_detail(
            self,
            id:str|uuid.UUID,
            type:str = "buyer",
            phone:str = None,
            zipcode:str = None,
            address:str = None,
            address_sub:str = None
    ):

        data = {
            "id": id,
            "type": type,
            "phone": phone,
            "zipcode": zipcode,
            "address": address,
            "address_sub": address_sub,
        }

        response = (
            supabase.table(self.table)
                    .insert(data)
                    .execute()
        )

        return response.data

    # 삭제되지 않은 회원 전체 조회 (Read: Deleted_at IS NULL)
    def get_active_users(self):
        response = (
            supabase.table(self.table)
                    .select("*")
                    .is_("deleted_at", "null")
                    .execute()
        )

        return response.data

    # 삭제되지 않은 단일 회원 조회
    def get_active_user_by_id(self, id: str):
        response = (
            supabase.table(self.table)
                    .select("*")
                    .eq("id", id)
                    .is_("deleted_at", "null")
                    .execute()
        )

        return response.data

    # 회원 정보 수정 (Update)
    def update_user_detail(self, id:str, **kwargs):
        kwargs["modified_at"] = datetime.now(timezone.utc).isoformat()
        response = (
            supabase.table(self.table)
                    .update(kwargs)
                    .eq("id", str(id))
                    .is_("deleted_at", "null")
                    execute()
        )

        return response.data

    # 소프트 삭제 (deleted_at 업데이트)
    def soft_delete_user(self, id:str|uuid.UUID):
        now = datetime.now(timezone.utc).isoformat()
        response = (
            supabase.table(self.table)
                    .update({"deleted_at":now})
                    .eq("id", str(user_id))
                    .is_("deleted_at", "null")
                    .execute()
        )
        return response.data